# ML-04 — Search Intelligence Data Contract

## 1. Unit of analysis + time window

**Lane 2: Refresh / Content Opportunity Scoring.** One row in the daily warehouse table represents one **content page × client × report date** observation. For this Week 3 contract, I will use the **March 2026** partition as the development/verification month. I will not use the June 2026 `_sample` for label development because it is the final month and should remain a sealed outcome/test window.

In [ ]:
# Query 1 — verify the stated grain for the March 2026 slice.
# Expected result: zero rows means no duplicate client × content × date records.

import duckdb
import os

HF_TOKEN = os.environ.get('HF_TOKEN')
if not HF_TOKEN:
    from google.colab import userdata
    HF_TOKEN = userdata.get('HF_TOKEN')

con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")
REL = 'hf://datasets/FlyRank/internship-warehouse'
DAILY = f"read_parquet('{REL}/fact_content_daily_performance/**/*.parquet')"

grain_check = con.sql(f"""
SELECT report_date, client_hash_id, content_hash_id, COUNT(*) AS row_count
FROM {DAILY}
WHERE report_date >= DATE '2026-03-01'
  AND report_date < DATE '2026-04-01'
GROUP BY 1, 2, 3
HAVING COUNT(*) > 1
LIMIT 5
""").df()
print('Duplicate grain rows:', len(grain_check))
grain_check

## 2. Fields: feature / label / context / excluded

- **Features:** earlier-window search/analytics measurements that are available before the ranking decision, such as prior impressions, clicks, sessions, and position summaries.
- **Label / proxy:** a decline outcome/proxy defined from a later outcome window. A label-derived field is never used as a feature.
- **Context:** `client_hash_id` and `content_hash_id` are identifiers used for joining, grouping, and grouped validation; they are not model features.
- **Excluded:** future-window measurements, `trend_direction`/`trend_pct` when they are used to derive the target, and product/private decision flags. They would leak information or encode the decision we are trying to support.

**Output:** a ranked queue of content pages for human review, with the priority driven by observable evidence.

**Availability rule:** analytics features are only considered available when the corresponding `ga4_data_available` flag is true; zero-filled GA4 values before availability are not treated as genuine zero engagement.

In [ ]:
# Query 2 — verify the March 2026 slice row count and date span.

slice_check = con.sql(f"""
SELECT
    COUNT(*) AS row_count,
    MIN(report_date) AS first_date,
    MAX(report_date) AS last_date
FROM {DAILY}
WHERE report_date >= DATE '2026-03-01'
  AND report_date < DATE '2026-04-01'
""").df()
slice_check

## 3. Verify it with queries + build five features

The three verification queries below check the grain, March slice size/date span, and GA4 availability. After those checks, the five-feature frame uses only information intended to be available before a later decision window.

In [ ]:
# Query 3 — verify GA4 availability using IS TRUE, as required by the contract.

availability_check = con.sql(f"""
SELECT COUNT(*) AS rows_with_ga4_available
FROM {DAILY}
WHERE report_date >= DATE '2026-03-01'
  AND report_date < DATE '2026-04-01'
  AND ga4_data_available IS TRUE
""").df()
availability_check

### Five-feature frame

For the five features, I use March 2026 page-level observations and aggregate the daily data within the month. The feature definitions are written so their availability is explicit:

1. **impressions_90d** — knowable at the decision moment because it summarizes search impressions observed before that decision.
2. **clicks_90d** — knowable at the decision moment because it summarizes clicks already observed before the decision.
3. **sessions_90d** — knowable at the decision moment because it summarizes observed analytics sessions, subject to the GA4 availability flag.
4. **avg_position** — knowable at the decision moment because it summarizes observed search position before the decision.
5. **content_age_days** — knowable at the decision moment because page age is determined from the content's existing history.

The final model should enforce a strict feature window before the outcome window; these are a contract-level feature sketch, not proof of causal usefulness.

In [ ]:
# Build a compact five-feature frame from the March 2026 slice.
# This is separate from the three verification queries above.

features = con.sql(f"""
SELECT
    client_hash_id,
    content_hash_id,
    SUM(gsc_impressions) AS impressions_90d,
    SUM(gsc_clicks) AS clicks_90d,
    SUM(CASE WHEN ga4_data_available IS TRUE THEN ga4_sessions ELSE 0 END) AS sessions_90d,
    AVG(NULLIF(gsc_avg_position, 0)) AS avg_position,
    MAX(content_age_days) AS content_age_days
FROM {DAILY}
WHERE report_date >= DATE '2026-03-01'
  AND report_date < DATE '2026-04-01'
GROUP BY 1, 2
""").df()

print('Feature-frame shape:', features.shape)
features.head(10)

### 4. The trap: deliberate label leakage

I will deliberately add a label-derived column to a small quick classification experiment. Because the feature contains the answer, a very high score is expected and is **not** evidence of a useful model. After demonstrating the trap, the leaked column is removed and the honest feature set is retained.

The important lesson is that a feature must be knowable before the prediction/review moment and must not be computed from the label or its outcome window.

In [ ]:
# Deliberate leakage experiment.
# We use the March feature frame and create a proxy only for demonstrating the trap.

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

leak_df = features.copy()
leak_df['decline_proxy'] = (leak_df['impressions_90d'] < leak_df['impressions_90d'].median()).astype(int)

# Intentionally leak the answer into a feature.
leak_df['LEAK_LABEL'] = leak_df['decline_proxy']

X_train, X_test, y_train, y_test = train_test_split(
    leak_df[['impressions_90d', 'clicks_90d', 'sessions_90d', 'avg_position', 'content_age_days', 'LEAK_LABEL']],
    leak_df['decline_proxy'], test_size=0.25, random_state=42, stratify=leak_df['decline_proxy']
)

leaky_model = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
leaky_model.fit(X_train.fillna(0), y_train)
leaky_pred = leaky_model.predict(X_test.fillna(0))
print(f'Leaky quick accuracy: {accuracy_score(y_test, leaky_pred):.3f}')

# Remove the leaked column and retain only honest features.
features_honest = features.copy()
print('LEAK_LABEL present after cleanup:', 'LEAK_LABEL' in features_honest.columns)
print('Honest feature columns:', list(features_honest.columns))

## Data limits

One important limitation is that the warehouse is an **unbalanced panel**: clients have different history depths. Therefore, a March row does not imply that every client has the same amount of usable history before March. GA4 availability also differs by client, so missing/zero-filled analytics values cannot be interpreted as comparable across all clients without using the availability flag.

The March slice is useful for development and verification, but it should not be treated as a final test. June 2026 is the final month and should remain sealed for final evaluation.

## Self-check

- [x] Unit of analysis and March 2026 development window stated.
- [x] Feature, label/proxy, context, and excluded fields classified.
- [x] Exactly three verification queries are provided: grain, row count/date span, and `IS TRUE` availability.
- [x] Five-feature frame provided with an availability explanation for each feature.
- [x] Label leakage is deliberately demonstrated and the leaked column is removed.
- [x] A concrete limitation of the slice is documented.
- [ ] Run all cells in Colab so the three warehouse query outputs and leakage result are genuinely recorded before committing.
